# Pokrycie kodu testami czarnoskrzynkowymi — Mealie

## Zadanie
Dla testów z zajęć 1 (`test_KacperS_zajecia1.py`) sprawdzić za pomocą `coverage.py`:

1. **jakie pokrycie kodu osiągnięto,**
2. **który przypadek testowy samodzielnie daje największe pokrycie,**
3. **czy są linie, które nie zostały wykonane** — jakie, i jakie przypadki dodać, by je pokryć.

Na końcu: wnioski.

## Zakres
Analiza dotyczy **trzech funkcji testowanych na zajęciach 1**:

| Funkcja | Plik |
|---|---|
| `create_recipe_slug` | `mealie/schema/recipe/recipe.py` |
| `QueryFilterBuilder.combine_filters` | `mealie/services/query_filter/builder.py` |
| `PlanRulesDay.from_date` | `mealie/schema/meal_plan/plan_rules.py` |

Dla pełnego obrazu pokazuję też — **tylko jako statystykę** — pokrycie całych plików oraz kodu wywoływanego pośrednio
(`combine_filters` waliduje filtr, uruchamiając parser z `builder.py`, `keywords.py`, `operators.py`).

## Jak mierzę
- **Pokrycie linii i gałęzi** (`branch=True`).
- **Wkład pojedynczego przypadku:** każdy przypadek (także każdy wariant `parametrize`) dostaje własny *kontekst*
  `coverage.py`, więc po jednym uruchomieniu wiadomo, które linie wykonał który test. Robi to `coverage_runner_zajecia2.py`.
- Testy uruchamiane z `--noconftest` (conftest ładuje całą aplikację i wymaga `python-ldap`).

## Przygotowanie

In [10]:
import json
import re
import subprocess
import sys
import tempfile
from collections import defaultdict
from html import escape
from pathlib import Path

from IPython.display import HTML, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "mealie").is_dir())
BB = ROOT / "tests" / "black_box"
RUNNER = BB / "coverage_runner_zajecia2.py"
SUITE = BB / "test_KacperS_zajecia1.py"
OUT = Path(tempfile.mkdtemp(prefix="cov_zajecia2_"))

# Testowane funkcje: nazwa -> (plik, nazwa w raporcie coverage, prefiks nazw testów)
TARGETS = {
    "create_recipe_slug":                 ("mealie/schema/recipe/recipe.py", "create_recipe_slug", "test_create_recipe_slug"),
    "QueryFilterBuilder.combine_filters": ("mealie/services/query_filter/builder.py", "QueryFilterBuilder.combine_filters", "test_combine_filters"),
    "PlanRulesDay.from_date":             ("mealie/schema/meal_plan/plan_rules.py", "PlanRulesDay.from_date", "test_from_date"),
}
TARGET_KEYS = {(f, qn) for f, qn, _ in TARGETS.values()}


def run_coverage(label: str, *test_files: Path) -> dict:
    """Uruchamia pytest pod coverage w osobnym procesie i wczytuje wyniki."""
    out = OUT / label
    proc = subprocess.run([sys.executable, str(RUNNER), str(out), *map(str, test_files)],
                          cwd=ROOT, capture_output=True, text=True, encoding="utf-8", errors="replace")
    if not (out / "coverage.json").exists():
        raise RuntimeError(proc.stdout[-3000:] + proc.stderr[-3000:])
    read = lambda name: json.loads((out / name).read_text(encoding="utf-8"))
    return {"cov": read("coverage.json"), "ctx": read("contexts.json"), "outcomes": read("outcomes.json")}


def files_of(run: dict) -> dict:
    return {k.replace("\\", "/"): v for k, v in run["cov"]["files"].items()}


def source_line(file: str, no: int) -> str:
    return (ROOT / file).read_text(encoding="utf-8").splitlines()[no - 1].strip()


def table(headers: list[str], rows: list[list]) -> None:
    th = "".join(f"<th style='border:1px solid #8884;padding:4px 8px;text-align:left'>{escape(h)}</th>" for h in headers)
    body = "".join("<tr>" + "".join(
        f"<td style='border:1px solid #8884;padding:4px 8px;vertical-align:top'>{c}</td>" for c in r) + "</tr>" for r in rows)
    display(HTML(f"<table style='border-collapse:collapse;font-size:13px'><tr>{th}</tr>{body}</table>"))


def bar(value: float, maximum: float, color: str = "#2f81f7") -> str:
    w = 0 if maximum == 0 else round(100 * value / maximum)
    return (f"<div style='background:#8882;width:140px;display:inline-block;vertical-align:middle'>"
            f"<div style='background:{color};width:{w}%;height:10px'></div></div>")


def pct(a: int, b: int) -> str:
    return f"{100 * a / b:.0f}%" if b else "—"


def fmt_lines(lines) -> str:
    """[3,4,5,9] -> '3–5, 9'"""
    out, start, prev = [], None, None
    for n in sorted(lines):
        if start is None:
            start = prev = n
        elif n == prev + 1:
            prev = n
        else:
            out.append((start, prev)); start = prev = n
    if start is not None:
        out.append((start, prev))
    return ", ".join(str(a) if a == b else f"{a}–{b}" for a, b in out) or "—"


def status_html(s: str) -> str:
    color = {"passed": "#1a7f37", "failed": "#cf222e"}.get(s, "#9a6700")
    return f"<b style='color:{color}'>{ {'passed': 'PASS', 'failed': 'FAIL'}.get(s, s.upper()) }</b>"

In [ ]:
run = run_coverage("zajecia1", SUITE)
FS = files_of(run)

o = list(run["outcomes"].values())
print(f"Testy: {len(o)} uruchomionych | {o.count('passed')} passed | {o.count('failed')} failed | {o.count('skipped')} skipped")

Testy: 32 uruchomionych | 26 passed | 6 failed | 0 skipped
(6 failed to znane rozbieżności z zajęć 1: KS-07, 09, 10, 19a, 19c, 28 — pokrycie liczy się niezależnie od wyniku testu)


---
## 1. Jakie pokrycie osiągnięto?
### 1a. Testowane funkcje

In [12]:
rows = []
for name, (f, qn, _) in TARGETS.items():
    fn = FS[f]["functions"][qn]
    s = fn["summary"]
    rows.append([f"<code>{qn}</code>", f"<code>{Path(f).name}</code>", s["num_statements"], s["covered_lines"],
                 f"<b>{pct(s['covered_lines'], s['num_statements'])}</b>",
                 f"{s['covered_branches']}/{s['num_branches']}" if s["num_branches"] else "brak gałęzi",
                 f"<b>{pct(s['covered_branches'], s['num_branches'])}</b>" if s["num_branches"] else "—",
                 fmt_lines(fn["missing_lines"])])
tot_s = sum(FS[f]["functions"][qn]["summary"]["num_statements"] for f, qn in TARGET_KEYS)
tot_c = sum(FS[f]["functions"][qn]["summary"]["covered_lines"] for f, qn in TARGET_KEYS)
tot_b = sum(FS[f]["functions"][qn]["summary"]["num_branches"] for f, qn in TARGET_KEYS)
tot_cb = sum(FS[f]["functions"][qn]["summary"]["covered_branches"] for f, qn in TARGET_KEYS)
rows.append(["<b>razem</b>", "", tot_s, tot_c, f"<b>{pct(tot_c, tot_s)}</b>", f"{tot_cb}/{tot_b}", f"<b>{pct(tot_cb, tot_b)}</b>", ""])
table(["Funkcja", "Plik", "Instrukcje", "Wykonane", "Linie %", "Gałęzie", "Gałęzie %", "Niewykonane linie"], rows)

Funkcja,Plik,Instrukcje,Wykonane,Linie %,Gałęzie,Gałęzie %,Niewykonane linie
create_recipe_slug,recipe.py,6,6,100%,4/4,100%,—
QueryFilterBuilder.combine_filters,builder.py,10,10,100%,4/4,100%,—
PlanRulesDay.from_date,plan_rules.py,4,4,100%,brak gałęzi,—,—
razem,,20,20,100%,8/8,100%,


### 1b. Dla porównania: całe pliki *(statystyka)*
Procent dla całego pliku jest niski, bo pliki zawierają wiele klas i metod, których testy z zajęć 1 nie dotyczyły
(np. `recipe.py` to cały model przepisu, `builder.py` — także budowanie zapytań SQL).

In [13]:
rows, tot = [], defaultdict(int)
for f, d in FS.items():
    s = d["summary"]
    for k in ("num_statements", "covered_lines", "num_branches", "covered_branches"):
        tot[k] += s[k]
    rows.append([f"<code>{f}</code>", s["num_statements"], s["covered_lines"], f"{s['percent_statements_covered']:.0f}%",
                 s["num_branches"], s["covered_branches"], pct(s["covered_branches"], s["num_branches"])])
rows.append(["<b>razem</b>", tot["num_statements"], tot["covered_lines"], pct(tot["covered_lines"], tot["num_statements"]),
             tot["num_branches"], tot["covered_branches"], pct(tot["covered_branches"], tot["num_branches"])])
table(["Plik", "Instrukcje", "Wykonane", "Linie %", "Gałęzie", "Wykonane", "Gałęzie %"], rows)

Plik,Instrukcje,Wykonane,Linie %,Gałęzie,Wykonane,Gałęzie %
mealie/schema/meal_plan/plan_rules.py,64,50,78%,0,0,—
mealie/schema/recipe/recipe.py,256,180,70%,42,4,10%
mealie/services/query_filter/builder.py,418,204,49%,208,74,36%
mealie/services/query_filter/keywords.py,82,33,40%,38,8,21%
mealie/services/query_filter/operators.py,20,20,100%,4,4,100%
razem,840,487,58%,292,90,31%


---
## 2. Który przypadek daje samodzielnie największe pokrycie?
Dla każdego przypadku liczę linie testowanych funkcji, które wykonał on **sam** (bez linii wykonywanych tylko przy
imporcie modułu, np. `def`):
- **własna funkcja:** jaką część funkcji, którą testuje, wykonał przypadek,
- **3 funkcje razem:** jaką część całego testowanego kodu,
- *kod pośredni:* liczba linii parsera wykonanych przy okazji (statystyka, tabela 1c).

In [15]:
SINGLE = {"test_create_recipe_slug_exactly_max_length_is_not_truncated": "KS-08",
          "test_combine_filters_single_filter": "KS-11",
          "test_combine_filters_ignores_empty_parts": "KS-17",
          "test_combine_filters_keeps_or_filter_grouped": "KS-18",
          "test_from_date_full_week_gives_seven_distinct_days": "KS-26",
          "test_from_date_does_not_depend_on_system_locale": "KS-28"}


def case_id(node: str) -> str:
    m = re.search(r"\[(KS-\d+[a-z]?)", node)
    return m.group(1) if m else SINGLE.get(node.split("::")[-1], node.split("::")[-1])


def case_title(node: str) -> str:
    m = re.search(r"\[KS-\d+[a-z]?-(.*)\]", node)
    name = node.split("::")[-1]
    return m.group(1).replace("-", " ") if m else re.sub(r"^test_(create_recipe_slug|combine_filters|from_date)_", "", name).replace("_", " ")


def target_of(node: str) -> str:
    test_name = node.split("::")[-1]
    return next(name for name, (_, _, prefix) in TARGETS.items() if test_name.startswith(prefix))


# instrukcje testowanych funkcji, które mógł wykonać test (bez linii wykonywanych tylko przy imporcie)
SCOPE = {}
for name, (f, qn, _) in TARGETS.items():
    fn = FS[f]["functions"][qn]
    SCOPE[name] = (set(fn["executed_lines"]) | set(fn["missing_lines"])) - (set(fn["executed_lines"]) - USED[f])
TOTAL = sum(len(v) for v in SCOPE.values())
IND_LINES = {(f, qn): set(FS[f]["functions"][qn]["executed_lines"]) | set(FS[f]["functions"][qn]["missing_lines"]) for f, qn in indirect}

ranking = []
for node, per in run["ctx"].items():
    lines = {f: set(v) for f, v in per.items()}
    covered = {name: lines.get(TARGETS[name][0], set()) & SCOPE[name] for name in TARGETS}
    own = target_of(node)
    ranking.append({
        "node": node, "id": case_id(node), "title": case_title(node), "fn": own, "status": run["outcomes"].get(node, "?"),
        "own": len(covered[own]), "own_total": len(SCOPE[own]),
        "all": sum(len(v) for v in covered.values()),
        "indirect": sum(len(lines.get(f, set()) & ls) for (f, _), ls in IND_LINES.items()),
    })
ranking.sort(key=lambda r: (-r["all"], -r["own"] / r["own_total"], -r["indirect"], r["id"]))

print(f"Testowany kod: {TOTAL} instrukcji ({', '.join(f'{n}: {len(s)}' for n, s in SCOPE.items())})")
mx = max(r["all"] for r in ranking)
table(["ID", "Przypadek", "Funkcja", "Własna funkcja", "3 funkcje razem", "", "Kod pośredni (linie)", "Wynik"],
      [[r["id"], escape(r["title"]), f"<code>{r['fn'].split('.')[-1]}</code>",
        f"{r['own']}/{r['own_total']} (<b>{pct(r['own'], r['own_total'])}</b>)",
        f"<b>{r['all']}</b>/{TOTAL} ({pct(r['all'], TOTAL)})", bar(r["all"], mx), r["indirect"], status_html(r["status"])]
       for r in ranking])

Testowany kod: 20 instrukcji (create_recipe_slug: 6, QueryFilterBuilder.combine_filters: 10, PlanRulesDay.from_date: 4)


ID,Przypadek,Funkcja,Własna funkcja,3 funkcje razem,,Kod pośredni (linie),Wynik
KS-17,ignores empty parts,combine_filters,8/10 (80%),8/20 (40%),,89,PASS
KS-13,three filters,combine_filters,7/10 (70%),7/20 (35%),,120,PASS
KS-18,keeps or filter grouped,combine_filters,7/10 (70%),7/20 (35%),,98,PASS
KS-19c,missing operator,combine_filters,7/10 (70%),7/20 (35%),,96,FAIL
KS-11,single filter,combine_filters,7/10 (70%),7/20 (35%),,89,PASS
KS-12,two filters,combine_filters,7/10 (70%),7/20 (35%),,89,PASS
KS-19a,missing value,combine_filters,7/10 (70%),7/20 (35%),,89,FAIL
KS-19b,unbalanced parenthesis,combine_filters,7/10 (70%),7/20 (35%),,73,PASS
KS-06,longer than default max length,create_recipe_slug,5/6 (83%),5/20 (25%),,0,PASS
KS-07,custom max length,create_recipe_slug,5/6 (83%),5/20 (25%),,0,FAIL


In [16]:
best = ranking[0]
print(f"Największe samodzielne pokrycie testowanego kodu: {best['id']} ({best['title']}) — "
      f"{best['all']} z {TOTAL} instrukcji ({pct(best['all'], TOTAL)}), "
      f"{best['own']}/{best['own_total']} własnej funkcji {best['fn']}\n")

print("Najlepszy przypadek dla każdej funkcji:")
for name in TARGETS:
    rs = [r for r in ranking if r["fn"] == name]
    top = max(r["own"] for r in rs)
    ids = [r["id"] for r in rs if r["own"] == top]
    print(f"  {name:<36} {top}/{len(SCOPE[name])} ({pct(top, len(SCOPE[name]))})  ->  {', '.join(ids)}")

top_ind = max(ranking, key=lambda r: r["indirect"])
print(f"\nNajwięcej kodu pośredniego (statystyka): {top_ind['id']} ({top_ind['title']}) — {top_ind['indirect']} linii parsera")

Największe samodzielne pokrycie testowanego kodu: KS-17 (ignores empty parts) — 8 z 20 instrukcji (40%), 8/10 własnej funkcji QueryFilterBuilder.combine_filters

Najlepszy przypadek dla każdej funkcji:
  create_recipe_slug                   5/6 (83%)  ->  KS-06, KS-07
  QueryFilterBuilder.combine_filters   8/10 (80%)  ->  KS-17
  PlanRulesDay.from_date               4/4 (100%)  ->  KS-28

Najwięcej kodu pośredniego (statystyka): KS-13 (three filters) — 120 linii parsera


---
## 3. Czy są niewykonane linie?
### 3a. Niewykonane linie i gałęzie w testowanych funkcjach

In [17]:
rows = []
for name, (f, qn, _) in TARGETS.items():
    fn = FS[f]["functions"][qn]
    for n in fn["missing_lines"]:
        rows.append([f"<code>{qn}</code>", f"linia {n}", f"<code>{escape(source_line(f, n))}</code>"])
    for a, b in fn["missing_branches"]:
        rows.append([f"<code>{qn}</code>", f"gałąź {a}→{b}", f"<code>{escape(source_line(f, a))}</code>"])
if rows:
    table(["Funkcja", "Miejsce", "Kod"], rows)
else:
    print("Brak: wszystkie linie i wszystkie gałęzie trzech testowanych funkcji zostały wykonane.")

Brak: wszystkie linie i wszystkie gałęzie trzech testowanych funkcji zostały wykonane.


---
## 4. Wnioski

### Odpowiedzi na pytania z zadania

| Pytanie | Odpowiedź |
|---|---|
| **1. Jakie pokrycie osiągnięto?** | Testowane funkcje: **100% linii (20/20) i 100% gałęzi (8/8)**. Dla porównania całe pliki: 58% linii i 31% gałęzi. Różnica wynika z tego, że pliki zawierają dużo kodu, którego testy nie dotyczyły. Kod parsera wywoływany pośrednio przez `combine_filters` wykonał się w 84% (statystyka). |
| **2. Który przypadek daje samodzielnie największe pokrycie?** | **KS-17** (`combine_filters` z `None` i `""` pomiędzy prawdziwymi filtrami): **8 z 20 instrukcji (40%)** testowanego kodu, 80% swojej funkcji. Wykonuje zarówno ścieżkę „pomiń pusty filtr", jak i „waliduj i dołącz filtr". W obrębie swoich funkcji najlepsze są KS-06 / KS-07 (83% `create_recipe_slug`) i KS-28 (100% `from_date`). W kodzie pośrednim najwięcej wykonuje KS-13 (120 linii parsera), bo zawiera listę i `CONTAINS ALL`. |
| **3. Czy są niewykonane linie?** | **Nie.** W trzech testowanych funkcjach wykonały się wszystkie linie i wszystkie gałęzie, więc żadnych przypadków nie trzeba dodawać. |

### Wnioski
1. **Zestaw testów z zajęć 1 jest kompletny pod względem pokrycia.** Podział na dane poprawne, brzegowe i niepoprawne dotarł do każdej ścieżki w każdej funkcji: obcinanie sluga, błąd pustej nazwy, pomijanie pustych filtrów, błąd walidacji i nieznany dzień tygodnia.
2. **100% pokrycia nie oznacza braku błędów.** Mimo pełnego pokrycia 6 z 32 przypadków kończy się FAIL (KS-07, 09, 10, 19a, 19c, 28). Pokrycie mówi tylko, *które linie się wykonały*, a nie *czy wynik był poprawny*. Błąd z KS-19c (filtr bez operatora przechodzi bez błędu) siedzi w linii pokrytej przez 8 przypadków.
3. **Niektóre linie są pokryte „cienko".** Dwie linie wykonuje tylko jeden przypadek (sekcja 3b):
   - `plan_rules.py:33` (`return PlanRulesDay.unset`) wykonuje tylko **KS-28**. Ten przypadek kończy się FAIL i zależy od ustawień systemu: na maszynie bez polskiego locale dostaje SKIP i linia przestaje być pokryta. Pełne pokrycie `from_date` jest więc „przypadkowe" i zależy od środowiska. Gdyby była potrzeba, można by dodać przypadek, który wywołuje tę ścieżkę bez zmiany locale.
   - `builder.py:228` (zamiana błędu walidacji na HTTP 400) wykonuje tylko **KS-19b**. Pozostałe dwa przypadki błędnej składni (KS-19a, KS-19c) do tej linii nie docierają, i właśnie dlatego są FAIL.
4. **Liczba przypadków nie przekłada się wprost na pokrycie.** 10 przypadków `create_recipe_slug` i 11 przypadków `from_date` sprawdza ważne zachowania, ale linii wykonują te same 2–5. Przypadki brzegowe i niepoprawne (np. KS-06, KS-09, KS-28) dokładają nowe linie, a kolejne „zwykłe" dane już nie. Ich wartość leży w sprawdzaniu wyników, nie w pokryciu.
5. **Pokrycie gałęzi jest tu tak samo pełne jak linii (8/8).** Każdy `if` / `for` został wykonany w obu kierunkach. Przy tak krótkich funkcjach to łatwiejsze niż w większym kodzie, ale potwierdza, że przypadki brzegowe zostały dobrane świadomie.
